# Test case — novelty, filter suggestions, preservation, LOMO vs Merkle break points
**Question:** a new bag arrives. Which bodies are *novel* compared with what we usually see? Where do they fall between our meshes?
Which filter material splits them **without destroying wings, scales and legs**? And across filters × hardware × capture methods, what actually improves the outcome?

Null hypothesis H0: the Track A kit stack (PERF-5 / GRID-4 / HEX-2 / SCREEN-1 / TRAY) is as good as any alternative for this bag.
H1: a novelty-driven filter choice beats the kit stack on intact recovery of the novel size band.

**Swap your data in the next cell.** Sample sizes come from baebugs measurement (AmScope) or a BugPicker `objects.jsonl`.
The reference set comes from past runs, the Essig prior or kit analogs.

In [ ]:
# ---- swap your data here ----
SAMPLE_CSV = None        # columns: length_mm,width_mm,profile   (None = built-in test bag)
REFERENCE_CSV = None     # columns: length_mm,width_mm           (None = kit analogs: rice/chia/pasta)
PRESERVATION = "ethanol_70"   # ethanol_70 | ethanol_95 | propylene_glycol | dry | frozen_thawed
N_EXPLORE = 120          # runs to append to the Merkle ledger
SEED = 7

In [ ]:
import sys, json, itertools, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, "..")
from baebugs.sieve import load_yaml
from baebugs.novelty import novel_sizes, mesh_gaps, suggest_filters
from baebugs.preservation import part_risks, gentlest_settings
from baebugs.explore import Ledger, simulate_run, lomo, compare_lomo_breakpoints, HARDWARE, CAPTURE
rng = np.random.default_rng(SEED)
MAT = {m['id']: m for m in load_yaml('materials.yaml')['materials']}

def bag():
    rows = []
    def add(n, L, W, prof):
        for _ in range(n): rows.append(dict(length_mm=rng.uniform(*L), width_mm=rng.uniform(*W), profile=prof))
    add(12, (5, 9),   (2.0, 3.0), 'hard_bodied')     # beetles
    add(10, (8, 14),  (2.9, 3.5), 'scaled_wings')    # small moths, wings folded  <- novel band
    add(10, (4, 7),   (0.8, 1.2), 'long_appendage')  # mosquitoes
    add(14, (1.5, 3), (0.7, 1.2), 'soft_bodied')     # aphids
    add(16, (2, 5),   (0.3, 0.6), 'membrane_wings')  # midges
    return pd.DataFrame(rows)
sample = pd.read_csv(SAMPLE_CSV) if SAMPLE_CSV else bag()
if REFERENCE_CSV:
    ref = pd.read_csv(REFERENCE_CSV)
else:  # kit analogs seen in previous runs: rice, chia, pasta, beetle-ish grains
    ref = pd.DataFrame(dict(length_mm=np.r_[rng.uniform(5,8,150), rng.uniform(1.6,2.2,150), rng.uniform(5,9,60), rng.uniform(5,8,40)],
                            width_mm =np.r_[rng.uniform(1.5,2.3,150), rng.uniform(1.0,1.4,150), rng.uniform(4.5,7,60), rng.uniform(2.0,3.0,40)]))
print(sample.groupby('profile').width_mm.describe()[['count','min','max']])

## 1 · Novel sizes (k-NN density vs the reference set)

In [ ]:
nov = pd.DataFrame(novel_sizes(sample.length_mm.values, sample.width_mm.values, ref.length_mm.values, ref.width_mm.values))
sample['novelty'] = nov.novelty.values; sample['novel'] = nov.novel.values
print(sample.groupby('profile').novel.agg(['sum','count']))
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(ref.length_mm, ref.width_mm, s=8, c='lightgray', label='reference')
for prof, g in sample.groupby('profile'):
    ax.scatter(g.length_mm, g.width_mm, s=[60 if n else 18 for n in g.novel], label=prof, edgecolors=['k' if n else 'none' for n in g.novel])
for m in ['GRID-4','HEX-2','SCREEN-1']:
    ax.axhline(MAT[m]['aperture_mm']*1.1, ls='--', lw=0.8, c='gray'); ax.text(13.5, MAT[m]['aperture_mm']*1.1, m, fontsize=7)
ax.set_xlabel('length (mm)'); ax.set_ylabel('width (mm) — governs mesh passage'); ax.set_yscale('log'); ax.legend(fontsize=7); ax.set_title('outlined = novel');

## 2 · Mesh gaps in the kit stack, and 3 · filter suggestions with preservation

In [ ]:
kit = [MAT[i]['aperture_mm'] for i in ['PERF-5','GRID-4','HEX-2','SCREEN-1']]
novel = sample[sample.novel]
gaps = mesh_gaps(novel.width_mm.tolist(), kit)
suggestions = {}
for g in gaps:
    profs = sample[sample.width_mm.between(*g['between_mm'])].profile.value_counts()
    prof = profs.index[0]
    print(f"\nGAP {g['between_mm']} mm  ({len(g['widths'])} novel bodies, mostly {prof}, rel gap {g['rel_gap']})")
    s = suggest_filters(g, prof, preservation=PRESERVATION)
    suggestions[tuple(g['between_mm'])] = (prof, s)
    print(pd.DataFrame(s)[['material','family','aperture_mm','in_kit','split_quality','retains','worst_part','damage_risk','score']].to_string(index=False))
    if s and s[0].get('gentle_settings'): print('  gentlest settings for top pick:', s[0]['gentle_settings'])

In [ ]:
# preservation view: worst-part damage risk per profile x material at default flow
mats = ['HEX-2','GRID-4','SCREEN-1','NET-MOSQ','NYLON-500','PRINT-SMOOTH-3']
risk = pd.DataFrame({m: {p: max(part_risks(p, MAT[m], preservation=PRESERVATION).values()) for p in load_yaml('materials.yaml')['specimen_profiles']} for m in mats})
print(risk.round(2))
plt.figure(figsize=(7,3)); plt.imshow(risk.values, cmap='Reds', vmin=0, vmax=1); plt.colorbar(label='worst-part damage risk')
plt.xticks(range(len(mats)), mats, rotation=30, fontsize=7); plt.yticks(range(len(risk)), risk.index, fontsize=7);

## 4 · Explore filters × hardware × capture with a Merkle ledger
Each run is a leaf; the root after each append is a break point. Runs arrive in random order, like a real weekend.
Target = the novel band from the first gap.

In [ ]:
lo, hi = gaps[0]['between_mm'] if gaps else (2.4, 4.4)
target_prof = suggestions[tuple(gaps[0]['between_mm'])][0] if gaps else 'scaled_wings'
bodies = [dict(width_mm=w, profile=p) for w, p in zip(sample.width_mm, sample.profile)]
filters = sorted({'GRID-4','HEX-2','SCREEN-1','PERF-5'} | {s['material'] for _, ss in suggestions.values() for s in ss if s['material'] in MAT})
space = [dict(filter=f, hardware=h, capture=c, flow=fl, preservation=PRESERVATION)
         for f, h, c, fl in itertools.product(filters, HARDWARE, CAPTURE, [0.25, 0.5, 1.0])]
order = rng.permutation(len(space))[:N_EXPLORE]
led = Ledger()
for i, j in enumerate(order):
    led.append(space[j], simulate_run(space[j], bodies, (lo, hi), seed=SEED + i))
print('runs:', len(led.runs), '| ledger verifies:', led.verify(), '| final root:', led.breakpoints[-1]['root'][:16])
gp = pd.DataFrame([dict(step=b['step'], root=b['root'][:12], delta=b['delta'], **b['best']['config'], score=b['best']['score']) for b in led.golden_path()])
print('\nGolden path (Merkle break points where the best config changed):'); print(gp.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
ax.plot([b['step'] for b in led.breakpoints], [b['best']['score'] for b in led.breakpoints], drawstyle='steps-post')
for b in led.golden_path(): ax.axvline(b['step'], c='orange', lw=0.6); ax.text(b['step'], b['best']['score'], b['best']['config']['filter'], fontsize=6, rotation=90)
ax.set_xlabel('run (Merkle leaf)'); ax.set_ylabel('best score so far'); ax.set_title('break points (orange) on the golden path');

## 5 · LOMO vs break points — do they agree on what matters?

In [ ]:
for f in ('filter','hardware','capture'):
    print(pd.DataFrame(lomo(led, f)).to_string(index=False)); print()
cmp = compare_lomo_breakpoints(led)
print(json.dumps(cmp, indent=1))

## 6 · Verdict on H0 and recommended stack

In [ ]:
kit_best = max((r for r in led.runs if r['config']['filter'] in ('GRID-4','HEX-2','PERF-5','SCREEN-1')), key=lambda r: r['score'])
new_best = max(led.runs, key=lambda r: r['score'])
print('best kit-only run :', kit_best['config'], kit_best['score'], kit_best['results'])
print('best overall run  :', new_best['config'], new_best['score'], new_best['results'])
print('H0 rejected (novelty-driven filter beats kit by >0.05):', new_best['score'] - kit_best['score'] > 0.05)
print('\nPer-profile gentlest settings on the winning filter:')
for p in sample.profile.unique():
    print(f"  {p:15s}", gentlest_settings(p, MAT[new_best['config']['filter']], PRESERVATION))
print('\nNOTE: results here come from simulate_run(); replace with verify-step counts from real runs.')